# Structured Output

---

## Use case

Les LLMs génèrent du texte libre, utile pour un humain, difficile à exploiter dans un programme.

Le **structured output** contraint le modèle à répondre dans un format défini (JSON, objet typé) validé par un schéma. Deux cas d'usage fondamentaux :

- **Génération**: produire un objet structuré depuis un prompt
- **Extraction**:  extraire des données structurées depuis du texte non structuré

Fil rouge de ce notebook : la **user story** agile.

## Stack

- **OpenAI SDK** (`jsr:@openai/openai`): `responses.parse` + Zod integration
- **Zod** (`jsr:@zod/zod`): validation de schéma TypeScript-first
- **AI SDK Vercel** (`npm:ai`, `npm:@ai-sdk/openai`): approche provider-agnostic en fin de notebook

## What's next

- `02-augmentation/function-calling`: structured output appliqué aux outils
- `03-agentique/use-cases`: structured output pour orchestrer des agents (Reflection pattern)

## Setup

**En local**
1. Copier `.env.example` en `.env` à la racine du repo
2. Renseigner `OPENAI_API_KEY`
3. Lancer le notebook avec le kernel Deno

**Sur Google Colab**
> ⚠️ Le kernel Deno n'est pas disponible nativement sur Colab. Ce notebook est prévu pour un environnement local.

In [ ]:
import { load } from "jsr:@std/dotenv";

const env = await load({ envPath: "../../.env" });
const apiKey = env["OPENAI_API_KEY"] ?? Deno.env.get("OPENAI_API_KEY");

if (!apiKey) throw new Error("OPENAI_API_KEY manquante — voir .env.example");

Client Openai

In [ ]:
import OpenAI from "jsr:@openai/openai";

const openai = new OpenAI({ apiKey });

## Pourquoi structured output ?

L'approche naïve: demander au modèle de répondre en JSON dans le prompt, puis `JSON.parse()`.

Les problèmes en pratique :
- **Réponse invalide**: le modèle entoure le JSON de markdown (` ```json ``` `), ajoute du texte avant/après
- **Clés hallucineées**: le modèle invente des champs non demandés, ou en omet
- **Types incohérents**: un champ attendu en array retourné en string
- **Parsing fragile**: un seul caractère invalide fait crasher `JSON.parse`

Le structured output résout ça en contraignant la génération au niveau du modèle, pas en post-processing. Le schéma est garanti, et non pas espéré.

## Génération

Produire une user story structurée depuis un prompt.

On définit le schéma attendu avec Zod, puis on passe ce schéma à `responses.parse`.
Le modèle est contraint à respecter la structure, `output_parsed` est directement utilisable.

In [ ]:
import { z } from "jsr:@zod/zod";

const UserStorySchema = z.object({
  title: z.string(),
  user_story: z.string().describe("Format: En tant que... je veux... afin de..."),
  acceptance_criteria: z.array(z.string()),
});

In [ ]:
import { zodTextFormat } from "jsr:@openai/openai/helpers/zod";

const generationResponse = await openai.responses.parse({
  model: "gpt-4.1-mini",
  input: [
    { role: "system", content: "Tu es un expert en rédaction de user stories agiles." },
    { role: "user", content: "Génère une user story pour une fonctionnalité de connexion via Google." },
  ],
  text: {
    format: zodTextFormat(UserStorySchema, "user_story"),
  },
});

// Réponse brute — le modèle ne fait pas de magie, c'est du JSON contraint
console.log("Réponse brute:", generationResponse);

In [ ]:
// Output parsé et typé  directement utilisable
const userStory = generationResponse.output_parsed;
console.log("Title:", userStory?.title);
console.log("User story:", userStory?.user_story);
console.log("Acceptance criteria:", userStory?.acceptance_criteria);

## Extraction

Extraire les éléments structurés d'une user story rédigée en langage naturel.

Même mécanique, prompt différent, le modèle lit le texte et remplit le schéma.
C'est le cas d'usage le plus fréquent en production : parser des emails, des tickets, des documents.

In [ ]:
const ExtractedUserStorySchema = z.object({
  actor: z.string().describe("Qui fait l'action"),
  action: z.string().describe("Ce que l'acteur veut faire"),
  benefit: z.string().describe("La valeur métier"),
  acceptance_criteria: z.array(z.string()),
});

In [ ]:
const rawUserStory = `
  En tant qu'utilisateur je voudrais pouvoir me connecter avec mon compte Google 
  pour ne pas avoir à créer un nouveau mot de passe et accéder plus rapidement 
  à l'application. La fonctionnalité devra proposer un bouton Google sur la page 
  de login, rediriger vers l'auth Google et créer le compte automatiquement si 
  c'est la première connexion.
`;

const extractionResponse = await openai.responses.parse({
  model: "gpt-4.1-mini",
  input: [
    { role: "system", content: "Tu es un expert en méthodes agiles. Extrais les éléments structurés de cette user story." },
    { role: "user", content: rawUserStory },
  ],
  text: {
    format: zodTextFormat(ExtractedUserStorySchema, "extracted_user_story"),
  },
});

// Réponse brute
console.log("Réponse brute:", extractionResponse);

In [ ]:
const extracted = extractionResponse.output_parsed;
console.log("Actor:", extracted?.actor);
console.log("Action:", extracted?.action);
console.log("Benefit:", extracted?.benefit);
console.log("Acceptance criteria:", extracted?.acceptance_criteria);

## Vers une approche provider-agnostic avec AI SDK

Même pattern avec AI SDK Vercel, `generateObject` remplace `responses.parse`.
Zod reste le même, seul le client change.

L'avantage : switcher vers Anthropic ou Mistral sans toucher au schéma ni à la logique.

In [ ]:
import { createOpenAI } from "npm:@ai-sdk/openai";
// import { createAnthropic } from "npm:@ai-sdk/anthropic";

const provider = createOpenAI({ apiKey });
const MODEL = provider("gpt-4.1-mini");

// Anthropic — décommenter et renseigner ANTHROPIC_API_KEY dans .env
// const anthropicProvider = createAnthropic({ apiKey: env["ANTHROPIC_API_KEY"] });
// const MODEL = anthropicProvider("claude-haiku-4-5");

### Génération

In [ ]:
import { generateObject } from "npm:ai";

const { object: generatedStory, usage } = await generateObject({
  model: MODEL,
  schema: UserStorySchema,
  system: "Tu es un expert en rédaction de user stories agiles.",
  prompt: "Génère une user story pour une fonctionnalité de connexion via Google.",
});

console.log(generatedStory);
console.log(`\nTokens — prompt: ${usage.promptTokens}, completion: ${usage.completionTokens}`);

### Extraction

In [ ]:
const { object: extractedStory } = await generateObject({
  model: MODEL,
  schema: ExtractedUserStorySchema,
  system: "Tu es un expert en méthodes agiles. Extrais les éléments structurés de cette user story.",
  prompt: rawUserStory,
});

console.log(extractedStory);